In [ ]:
"""
fjord_profiles.py
-----------------
Plots fjord CTD profile summaries (temperature, salinity, TS diagram, bathymetry).
Uses config_loader for data paths; BedMachine & shapefiles remain hardcoded below.
Only processes 2016 profiles.
"""

import os
import traceback

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import netCDF4
import gsw
import xarray as xr
import geopandas as gpd
import cartopy.crs as ccrs
from cartopy.mpl.geoaxes import GeoAxes
from matplotlib.lines import Line2D
from pyproj import CRS, Transformer
from shapely.geometry import box, Polygon, LineString, MultiPolygon
from shapely.ops import unary_union

# ── Config loader ──────────────────────────────────────────────────────────────
from config_loader import load_paths

paths       = load_paths()
csv_dir     = paths["csv_dir"]
nc_dir      = paths["nc_dir"]
results_root = paths["results_dir"]

output_dir = results_root / "caseStudies_qimKlu"
output_dir.mkdir(parents=True, exist_ok=True)

qim_csv = csv_dir / "qim_fjord_data.csv"
klu_csv = csv_dir / "klu_fjord_data.csv"

# ── Hardcoded paths (BedMachine + shapefiles) ─────────────────────────────────
BEDMACHINE_PATH = (
    r"C:\Users\efc4\OneDrive - University of St Andrews\Desktop\PhD"
    r"\First_Year\Projects\Fjord_Shelf_Observations\Working_data"
    r"\Bathymetry\BedMachineGreenland-v5.nc"
)

_BASE = (
    r"C:\Users\efc4\OneDrive - University of St Andrews\Desktop\PhD"
    r"\First_Year\Projects\Fjord_Shelf_Observations\Working_data"
)
SHAPEFILE_DEFS = [
    # (path, label, colour, zorder, fill)
    (
        _BASE + r"\GEEDiT_terminus_lines\SE\GEEDiT_termini_SE_all.gpkg",
        "Terminus Line", "blue", 4, False,
    ),
    (
        _BASE + r"\Gerrish_2020_Greenland_coast\Gerrish_2020_Greenland_coast.gpkg",
        "Land", "grey", 2, True,
    ),
    (
        _BASE + r"\PROMICE2022IceMask\02-PROMICE-2022-IceMask-polygon.gpkg",
        "Ice Sheet", "white", 3, True,
    ),
]

# ── Global plot style ──────────────────────────────────────────────────────────
plt.rcParams.update({
    "xtick.labelsize": 14,
    "ytick.labelsize": 14,
    "legend.fontsize": 14,
})

# ══════════════════════════════════════════════════════════════════════════════
# 1.  DATA LOADING
# ══════════════════════════════════════════════════════════════════════════════

def load_fjord_data(csv_file: str | os.PathLike) -> pd.DataFrame | None:
    """Load fjord metadata CSV and ensure fjord_id is a string."""
    try:
        df = pd.read_csv(csv_file)
        df["fjord_id"] = df["fjord_id"].astype(str)
        return df
    except Exception as exc:
        print(f"Error loading CSV: {exc}")
        return None


def filter_2016(df: pd.DataFrame) -> pd.DataFrame:
    """
    Keep only rows whose date column contains '2016'.
    Tries a 'date' column first, then falls back to 'year'.
    """
    if "date" in df.columns:
        return df[df["date"].astype(str).str.contains("2016")]
    if "year" in df.columns:
        return df[df["year"].astype(str) == "2016"]
    raise KeyError("No 'date' or 'year' column found to filter 2016 profiles.")


def match_profile_pairs(fjord_data: pd.DataFrame):
    """
    Match near-glacier (NG) profiles with their mouth counterparts.

    Returns
    -------
    matched_pairs : list of (ng_row, mouth_profiles_df)
    failed_ids    : list of icepicks_ids with no mouth match
    """
    ng_profiles  = fjord_data[fjord_data["type"].str.lower() == "near_glacier"]
    matched_pairs, failed_ids = [], []

    for _, ng_row in ng_profiles.iterrows():
        fid = ng_row["fjord_id"]
        mouth = fjord_data[
            (fjord_data["fjord_id"] == fid)
            & (fjord_data["type"].str.lower() == "fjord_mouth")
        ]
        if mouth.empty:
            print(f"No mouth profile for fjord_id: {fid}")
            failed_ids.append(fid)
        else:
            matched_pairs.append((ng_row, mouth))

    return matched_pairs, failed_ids


# ══════════════════════════════════════════════════════════════════════════════
# 2.  PROFILE PROCESSING
# ══════════════════════════════════════════════════════════════════════════════

def isolate_descent_phase(depth: np.ndarray, values: np.ndarray):
    """Return only the downcast data."""
    deepest_idx = np.argmax(depth)
    return depth[: deepest_idx + 1], values[: deepest_idx + 1]


def load_nc_profile(nc_path: str):
    """Load depth, temperature, and salinity from a GSW NetCDF CTD file."""
    with netCDF4.Dataset(nc_path) as ds:
        depth = np.squeeze(ds.variables["depth"][:])
        temp  = np.squeeze(ds.variables["conservative_temperature"][:])
        sal   = np.squeeze(ds.variables["absolute_salinity"][:])

    depth = np.ma.masked_invalid(depth).filled(np.nan)
    temp  = np.ma.masked_invalid(temp).filled(np.nan)
    sal   = np.ma.masked_invalid(sal).filled(np.nan)
    return depth, temp, sal


def bin_profile(depth: np.ndarray, values: np.ndarray, bin_edges: np.ndarray):
    """
    Bin profile data into fixed depth intervals using mean per bin.
    Requires at least 3 valid points per bin; otherwise returns NaN.
    Returns (bin_centers, binned_values).
    """
    bin_centers = (bin_edges[:-1] + bin_edges[1:]) / 2
    binned = np.full(len(bin_centers), np.nan)

    for i in range(len(bin_edges) - 1):
        mask = (
            (depth >= bin_edges[i]) &
            (depth <  bin_edges[i + 1]) &
            np.isfinite(values)
        )
        if mask.sum() >= 3:
            binned[i] = np.nanmean(values[mask])

    return bin_centers, binned


def load_fjord_profiles(csv_path, nc_dir, bin_edges=np.arange(1, 1001, 10)):
    """
    Load, quality-control, and bin all profiles for a single fjord CSV.

    Returns
    -------
    profiles : dict keyed by type string ('near_glacier', 'fjord_mouth', 'mid_fjord'),
               each value a dict with keys 'depth', 'temp', 'sal', 'label', 'style', 'row'.
    """
    VALID_TYPES = {"fjord_mouth", "mid_fjord", "near_glacier"}

    _STYLE = {
        "near_glacier": ("Near-glacier", "r-"),
        "fjord_mouth":  ("Mouth",        "k-"),
        "mid_fjord":    ("Mid-fjord",    "-"),
    }

    df = pd.read_csv(csv_path)
    df["type"] = df["type"].str.strip().str.lower()
    df = df[(df["year"] == 2016) & df["type"].isin(VALID_TYPES)]

    profiles = {}

    for _, row in df.iterrows():
        nc_path = os.path.join(nc_dir, row["GSW_filename"])
        if not os.path.exists(nc_path):
            print(f"Missing file: {row['GSW_filename']}")
            continue

        try:
            depth, temp, sal = load_nc_profile(nc_path)

            # Isolate descent phase
            depth, temp = isolate_descent_phase(depth, temp)
            _,     sal  = isolate_descent_phase(depth, sal)

            # Salinity sanity check — propagate mask to temperature
            sal[(sal < 30) | (sal > 40)] = np.nan
            temp[np.isnan(sal)]          = np.nan

            # Bin
            depth_bins, temp_binned = bin_profile(depth, temp, bin_edges)
            _,           sal_binned = bin_profile(depth, sal,  bin_edges)

            ptype = row["type"]
            label, style = _STYLE[ptype]

            profiles[ptype] = {
                "depth": depth_bins,
                "temp":  temp_binned,
                "sal":   sal_binned,
                "label": label,
                "style": style,
                "row":   row,          # carry metadata for TS diagram / bathy
            }

        except Exception as exc:
            print(f"Failed to load {nc_path}: {exc}")

    return profiles


# ══════════════════════════════════════════════════════════════════════════════
# 3.  BEDMACHINE CACHE
# ══════════════════════════════════════════════════════════════════════════════

class BedMachineCache:
    """Loads BedMachine once and serves spatial subsets on request."""

    def __init__(self, path: str):
        self.ds  = xr.open_dataset(
            path, decode_coords="all",
            drop_variables=["errbed", "source"],
        )
        self.bed = self.ds["bed"]

    def get_map_extent(
        self, x_min, x_max, y_min, y_max,
        coarsen: int = 10,           # ← increased from 5 for faster rendering
    ):
        return (
            self.bed
            .sel(x=slice(x_min, x_max), y=slice(y_max, y_min))
            .coarsen(x=coarsen, y=coarsen, boundary="trim")
            .mean()
            .compute()
        )


# Load once at module level so every fjord reuses the same object.
bed_cache = BedMachineCache(BEDMACHINE_PATH)


def subset_bedmachine(cache, ng_coords, df_coords, buffer_km=50_000):
    """Transform lat/lon → polar stereo, then extract a BedMachine tile."""
    wgs84  = CRS.from_epsg(4326)
    ps3413 = CRS.from_epsg(3413)
    tf     = Transformer.from_crs(wgs84, ps3413, always_xy=True)

    x_ng, y_ng = tf.transform(*ng_coords)
    x_df, y_df = tf.transform(*df_coords)

    x_min = min(x_df, x_ng) - buffer_km
    x_max = max(x_df, x_ng) + buffer_km
    y_min = min(y_df, y_ng) - buffer_km
    y_max = max(y_df, y_ng) + buffer_km

    return cache.get_map_extent(x_min, x_max, y_min, y_max)


# ══════════════════════════════════════════════════════════════════════════════
# 4.  PLOTTING HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _delta_inset(ax, depth_a, vals_a, depth_b, vals_b, xlim, title, xlabel, span_positive=True):
    """
    Generic helper: plot (vals_a − vals_b) vs depth in an inset axes.
    Shades the 'anomaly' side of zero in red (positive) or blue (negative).
    """
    mask_a = np.isfinite(vals_a) & (vals_a <= 50)
    mask_b = np.isfinite(vals_b) & (vals_b <= 50)
    if not (mask_a.any() and mask_b.any()):
        return

    common_min = max(depth_a[mask_a].min(), depth_b[mask_b].min())
    common_max = min(depth_a[mask_a].max(), depth_b[mask_b].max())
    if common_min >= common_max:
        return

    common = np.linspace(common_min, common_max, 200)
    da = np.interp(common, depth_a[mask_a], vals_a[mask_a])
    db = np.interp(common, depth_b[mask_b], vals_b[mask_b])
    delta = da - db

    inset = ax.inset_axes([0.65 if span_positive else 0.15, 0.15, 0.3, 0.3])
    inset.plot(delta, common, "m-")
    inset.set_xlim(xlim)
    inset.set_ylim([1000, 0])

    if span_positive:
        inset.axvspan(0, xlim[1], color="red",  alpha=0.1)
    else:
        inset.axvspan(xlim[0], 0, color="blue", alpha=0.1)

    inset.set_title(title,   fontsize=14)
    inset.set_xlabel(xlabel, fontsize=12)
    inset.set_ylabel("Depth (m)", fontsize=12)
    inset.tick_params(labelsize=12)
    inset.grid(False)


def melt_temperature(S_melt: float) -> float:
    """Effective meltwater temperature (°C) following the three-equation approach."""
    L, c_w, c_i = 335_000, 3974, 2009
    T_i_K      = -15 + 273.15
    Theta_f_K  = gsw.CT_freezing(S_melt, p=0, saturation_fraction=0) + 273.15
    return (Theta_f_K - L / c_w - (c_i / c_w) * (Theta_f_K - T_i_K)) - 273.15


def _plot_gade_lines(ax, S_farfield_vals, Theta_farfield):
    """Overplot a family of Gade (subglacial-melt) lines."""
    Theta_melt = melt_temperature(0.0)
    for S_ff in S_farfield_vals:
        S   = np.linspace(0.0, S_ff, 100)
        m   = (Theta_farfield - Theta_melt) / (S_ff - 0.0)
        T   = Theta_melt + m * (S - 0.0)
        ax.plot(S, T, linestyle="--", color="orange", alpha=0.5, linewidth=1)


def _plot_freshwater_mixing_lines(ax, lat, lon, p_grounding,
                                  seawater_salinity=36,
                                  seawater_temps=(0, 2, 4, 6, 8, 10)):
    """Freshwater mixing lines from subglacial runoff to ambient seawater."""
    theta_sub = gsw.CT_freezing(0, p=p_grounding, saturation_fraction=0)
    S_fw  = np.linspace(seawater_salinity, 0, 100)
    for theta_ff in seawater_temps:
        m = (theta_ff - theta_sub) / (seawater_salinity - 0)
        T = theta_sub + m * (S_fw - 0)
        ax.plot(S_fw, T, color="blue", linestyle="--", linewidth=1, alpha=0.2)


# ══════════════════════════════════════════════════════════════════════════════
# 5.  MAIN PLOT FUNCTIONS
# ══════════════════════════════════════════════════════════════════════════════

def plot_temperature_salinity_profiles(
    profiles,
    icepicks_ids,
    ng_depth=None, ng_temperature=None, ng_salinity=None,
    mouth_depth=None, mouth_temperature=None, mouth_salinity=None,
    sill_depth=np.nan,
    grounding_depth=np.nan,
):
    """Four-panel figure: temperature profile, salinity profile, TS diagram, bathymetry."""
    fig, axes = plt.subplots(nrows=2, ncols=2, figsize=(16, 20))

    for ax, lbl in zip(axes.flat, ["a", "b", "c", "d"]):
        ax.text(-0.15, 1.02, f"({lbl})", transform=ax.transAxes,
                fontsize=18, fontweight="bold", va="top", ha="left")

    fig.subplots_adjust(top=0.85, bottom=0.30, hspace=0.25, wspace=0.25)

    # ── Draw in order: mouth → mid_fjord → near_glacier
    DRAW_ORDER = ["fjord_mouth", "mid_fjord", "near_glacier"]
    MIDDLE_COLOUR = "orange"

    for ptype in DRAW_ORDER:
        p = profiles.get(ptype)
        if p is None:
            continue
        d, t, s = p["depth"], p["temp"], p["sal"]
        lbl, sty = p["label"], p["style"]
        kw = dict(color=MIDDLE_COLOUR) if ptype == "mid_fjord" else {}
        axes[0, 0].plot(t, d, sty, label=lbl, **kw)
        axes[0, 1].plot(s, d, sty, label=lbl, **kw)

    # ── Horizontal reference lines
    for ax_ref in (axes[0, 0], axes[0, 1]):
        if not np.isnan(sill_depth):
            ax_ref.axhline(sill_depth,      color="green", linestyle="--")
        if not np.isnan(grounding_depth):
            ax_ref.axhline(grounding_depth, color="blue",  linestyle="-.")

    # ── Temperature axes
    axes[0, 0].invert_yaxis()
    axes[0, 0].set_xlim([-2, 8]);  axes[0, 0].set_ylim([1000, 0])
    axes[0, 0].set_xlabel("Temperature (°C)", fontsize=16)
    axes[0, 0].set_ylabel("Depth (m)",        fontsize=16)
    axes[0, 0].grid(False)

    # ── Salinity axes
    axes[0, 1].invert_yaxis()
    axes[0, 1].set_xlim([32, 35.3]); axes[0, 1].set_ylim([1000, 0])
    axes[0, 1].set_xlabel("Salinity (g/kg)", fontsize=16)
    axes[0, 1].set_ylabel("Depth (m)",       fontsize=16)
    axes[0, 1].grid(False)

    # ── Shared legend (deduped)
    handles, labels = axes[0, 0].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    for ax_lg in (axes[0, 0], axes[0, 1]):
        ax_lg.legend(by_label.values(), by_label.keys(), loc="upper right")

    # ── ΔT inset
    if ng_depth is not None and mouth_depth is not None:
        _delta_inset(
            axes[0, 0],
            ng_depth, ng_temperature,
            mouth_depth, mouth_temperature,
            xlim=(-4, 4), title="ΔT (NG − Mouth)", xlabel="ΔT (°C)",
            span_positive=True,
        )

    # ── ΔS inset
    if ng_depth is not None and mouth_depth is not None:
        _delta_inset(
            axes[0, 1],
            ng_depth, ng_salinity,
            mouth_depth, mouth_salinity,
            xlim=(-1, 1), title="ΔS (NG − Mouth)", xlabel="ΔS (g/kg)",
            span_positive=False,
        )

    return fig, axes


def plot_ts_diagram(
    ax,
    ng_depth, ng_temperature, ng_salinity,
    mouth_depth, mouth_temperature, mouth_salinity,
    middle_depth=None, middle_temperature=None, middle_salinity=None,
    fjord_row=None,
):
    """TS diagram with density contours, water-mass labels, Gade & mixing lines."""

    # ── Density contours
    S_grid, T_grid = np.meshgrid(np.linspace(32, 35.3, 100), np.linspace(-2, 5, 100))
    D = gsw.sigma0(S_grid, T_grid)

    levels = np.arange(np.floor(D.min()), np.ceil(D.max()) + 0.5, 0.5)
    CS = ax.contour(S_grid, T_grid, D, levels=levels,
                    colors="black", linewidths=0.5, zorder=0)
    ax.clabel(CS, inline=True, fontsize=12, fmt="%.1f", zorder=0)

    dsow = ax.contour(S_grid, T_grid, D, levels=[27.8], colors="black", linewidths=1.5)
    ax.clabel(dsow, inline=True, fontsize=12, fmt="%1.1f")

    # ── AWm clipped polygon
    contour_path     = dsow.collections[0].get_paths()[0].vertices
    contour_line     = LineString(contour_path)
    awm_box = Polygon([
        (34.565, 0.981), (34.967, 0.981),
        (34.967, 4.470), (34.565, 4.470),
    ])
    clip_bottom = Polygon(
        list(contour_path) + [(35.5, -10), (32, -10), (32, contour_path[0, 1])]
    )
    awm_clipped = awm_box.intersection(clip_bottom)
    if not awm_clipped.is_empty:
        polys = [awm_clipped] if awm_clipped.geom_type == "Polygon" else list(awm_clipped.geoms)
        for poly in polys:
            ax.add_patch(patches.Polygon(
                list(poly.exterior.coords),
                facecolor="yellow", edgecolor="yellow", alpha=0.3, zorder=1,
            ))

    # ── Water-mass labels
    for txt, x, y, colour in [
        ("AW",   35.1,  4.75, "orange"),
        ("DSOW", 35.1,  1.00, "navy"),
        ("AWm",  34.766, 3.5, "orange"),
        ("PSW",  33.6,  0.2,  "blue"),
        ("PSWw", 32.7,  3.0,  "cyan"),
    ]:
        ax.text(x, y, txt, color=colour, ha="center", va="center",
                fontsize=14, zorder=2)

    # ── Depth-marker helper
    def _depth_markers(depths, sal, temp, colour, marker_depths=([30, 130, 200, 300, 400])):
        if any(v is None for v in (depths, sal, temp)):
            return
        for d in marker_depths:
            idx = np.argmin(np.abs(depths - d))
            s, t = sal[idx], temp[idx]
            if np.isfinite(s) and np.isfinite(t):
                ax.scatter(s, t, color=colour, marker="x", s=150, zorder=10)
                off = 0.15 if colour == "red" else -0.15
                va  = "bottom" if colour == "red" else "top"
                ha  = "right"  if colour == "red" else "left"
                ax.text(s, t + off, f"{d}m", ha=ha, va=va,
                        fontsize=12, color=colour)

    # ── Data scatter
    if mouth_salinity is not None:
        ax.plot(mouth_salinity, mouth_temperature, "ko", markersize=5, label="Mouth")
        _depth_markers(mouth_depth, mouth_salinity, mouth_temperature, "black")

    if middle_salinity is not None:
        ax.plot(middle_salinity, middle_temperature, "o", color="orange",
                markersize=5, label="Mid-fjord")
        _depth_markers(middle_depth, middle_salinity, middle_temperature, "orange")

    if ng_salinity is not None:
        ax.plot(ng_salinity, ng_temperature, "ro", markersize=5, label="Near-glacier")
        _depth_markers(ng_depth, ng_salinity, ng_temperature, "red")

    # ── Gade & freshwater mixing lines
    _plot_gade_lines(ax, S_farfield_vals=np.arange(26, 40, 1), Theta_farfield=10)

    if fjord_row is not None:
        lat     = fjord_row["latitude"]
        depth_m = -fjord_row["grounding_line_depth_m"]
        p_grnd  = gsw.p_from_z(depth_m, lat)
        _plot_freshwater_mixing_lines(ax, lat=lat, lon=fjord_row["longitude"],
                                      p_grounding=p_grnd)

    ax.set_xlim(32, 35.3);  ax.set_ylim(-2, 5)
    ax.set_xlabel("Absolute Salinity (g/kg)",     fontsize=14)
    ax.set_ylabel("Conservative Temperature (°C)", fontsize=14)
    ax.grid(False)
    ax.legend(loc="upper left")


def plot_bathymetry_panel(fig, ax, bed_subset, df_profile, fjord_row,
                          middle_row, shapefile_defs):
    """
    Replace axes[1,1] with a polar-stereo bathymetry map + Greenland inset.

    Shapefiles are rasterised via `rasterized=True` on the pcolormesh and by
    drawing vector geometries at reduced resolution (simplify_tolerance).
    """
    SIMPLIFY_M = 500   # metres — simplify shapefile geometry for faster rendering

    tf = Transformer.from_crs(CRS.from_epsg(4326), CRS.from_epsg(3413), always_xy=True)
    x_df,  y_df  = tf.transform(df_profile["longitude"],  df_profile["latitude"])
    x_ng,  y_ng  = tf.transform(fjord_row["longitude"],   fjord_row["latitude"])
    x_mid, y_mid = tf.transform(middle_row["longitude"],  middle_row["latitude"])

    BUF = 50_000
    x_min = min(x_df, x_ng, x_mid) - BUF;  x_max = max(x_df, x_ng, x_mid) + BUF
    y_min = min(y_df, y_ng, y_mid) - BUF;  y_max = max(y_df, y_ng, y_mid) + BUF

    # ── Replace placeholder axes
    pos = ax.get_position()
    fig.delaxes(ax)
    ax_b = fig.add_axes(pos.bounds, projection=ccrs.NorthPolarStereo())
    ax_b.text(-0.15, 0.98, "(d)", transform=ax_b.transAxes,
              fontsize=18, fontweight="bold", va="top", ha="left")

    # ── Bathymetry (rasterized=True prevents per-pixel vector overhead in PDF/SVG)
    bathy = ax_b.pcolormesh(
        bed_subset.x, bed_subset.y, bed_subset,
        cmap="terrain", shading="auto",
        transform=ccrs.NorthPolarStereo(),
        zorder=0,
        rasterized=True,          # ← key speed-up for vector-format saves
    )
    cbar = fig.colorbar(bathy, ax=ax_b, orientation="vertical", fraction=0.02, pad=0.04)
    cbar.set_label("Depth (m)", fontsize=14)

    ax_b.set_extent([x_min, x_max, y_min, y_max], crs=ccrs.NorthPolarStereo())

    # ── Scale bar
    s0 = (x_min + 0.05 * (x_max - x_min), y_min + 0.05 * (y_max - y_min))
    ax_b.add_artist(Line2D(
        [s0[0], s0[0] + 20_000], [s0[1], s0[1]],
        color="black", linewidth=3, transform=ccrs.NorthPolarStereo(), zorder=20,
    ))
    ax_b.text(s0[0] + 10_000, s0[1] + 0.02, "20 km",
              ha="center", va="bottom", fontsize=14, color="black",
              transform=ccrs.NorthPolarStereo(), zorder=20)

    # ── CTD scatter
    for x, y, colour, lbl in [
        (x_df,  y_df,  "black",  "Mouth"),
        (x_ng,  y_ng,  "red",    "Near glacier"),
        (x_mid, y_mid, "orange", "Mid-fjord"),
    ]:
        ax_b.scatter(x, y, color=colour, marker="*", s=200,
                     transform=ccrs.NorthPolarStereo(), zorder=10, label=lbl)

    # ── Rasterised shapefiles
    extent_box = box(x_min, y_min, x_max, y_max)
    for path, label, colour, zorder, fill in shapefile_defs:
        gdf = gpd.read_file(path).to_crs(epsg=3413)
        gdf = gdf[gdf.intersects(extent_box)]
        if gdf.empty:
            continue
        # Simplify geometry to reduce vertex count → faster rendering
        gdf["geometry"] = gdf["geometry"].simplify(SIMPLIFY_M, preserve_topology=True)
        ax_b.add_geometries(
            gdf.geometry,
            crs=ccrs.NorthPolarStereo(),
            facecolor=colour if fill else "none",
            edgecolor=colour,
            linewidth=1,
            zorder=zorder,
            label=label,
            rasterized=True,       # ← rasterise vector fills
        )

    # ── Greenland inset
    bpos   = ax_b.get_position()
    iw, ih = 0.30 * bpos.width, 0.30 * bpos.height
    ax_i   = fig.add_axes(
        [bpos.x0 - 0.0185, bpos.y1 - ih, iw, ih],
        projection=ccrs.NorthPolarStereo(),
    )

    land_path = next(p for p, lbl, *_ in shapefile_defs if "coast" in p.lower())
    gdf_land  = gpd.read_file(land_path).to_crs(epsg=3413)
    minx, miny, maxx, maxy = gdf_land.total_bounds
    pad = 200e3
    ax_i.set_extent([minx - pad, maxx + pad, miny - pad, maxy + pad],
                    crs=ccrs.NorthPolarStereo())

    for path, label, colour, zorder, fill in shapefile_defs:
        if "coast" in path.lower() or "icemask" in path.lower():
            gdf = gpd.read_file(path).to_crs(epsg=3413)
            gdf["geometry"] = gdf["geometry"].simplify(SIMPLIFY_M, preserve_topology=True)
            ax_i.add_geometries(
                gdf.geometry,
                crs=ccrs.NorthPolarStereo(),
                facecolor=colour if fill else "none",
                edgecolor=colour,
                linewidth=0.5,
                zorder=zorder,
                rasterized=True,
            )

    ax_i.scatter(x_ng, y_ng, color="red", marker="*", s=80,
                 transform=ccrs.NorthPolarStereo(), zorder=10)

    for spine in ax_i.spines.values():
        spine.set_visible(True);  spine.set_color("grey");  spine.set_linewidth(0.5)

    return ax_b


# ══════════════════════════════════════════════════════════════════════════════
# 6.  MAIN LOOP
# ══════════════════════════════════════════════════════════════════════════════

def log(msg: str):
    print(f"[LOG] {msg}", flush=True)


def process_csv(csv_file, data_dir, output_dir, bed_cache, shapefile_defs):
    """Run the full pipeline for one CSV file (qim or klu)."""
    log(f"Loading {csv_file} …")
    fjord_data = load_fjord_data(csv_file)
    if fjord_data is None:
        return

    log("Filtering to 2016 profiles …")
    fjord_data = filter_2016(fjord_data)
    log(f"  {len(fjord_data)} rows after 2016 filter.")

    log("Matching NG ↔ Mouth pairs …")
    pairs, failed = match_profile_pairs(fjord_data)
    log(f"  {len(pairs)} matched, {len(failed)} failed.")

    for i, (ng_row, mouth_df) in enumerate(pairs, 1):
        fid = ng_row["fjord_id"]
        log(f"\n=== Fjord {i}/{len(pairs)}  (id={fid}) ===")

        try:
            log("  Loading and binning NetCDF files …")
            profiles = load_fjord_profiles(csv_file, data_dir)

            ng    = profiles.get("near_glacier", {})
            mouth = profiles.get("fjord_mouth",  {})
            mid   = profiles.get("mid_fjord",    {})

            ng_d,    ng_T,    ng_S    = ng.get("depth"),    ng.get("temp"),    ng.get("sal")
            mouth_d, mouth_T, mouth_S = mouth.get("depth"), mouth.get("temp"), mouth.get("sal")
            mid_d,   mid_T,   mid_S   = mid.get("depth"),   mid.get("temp"),   mid.get("sal")

            sill_d = ng_row.get("sill_depth_m",          np.nan)
            grnd_d = ng_row.get("grounding_line_depth_m", np.nan)

            log("  Plotting profiles …")
            fig, axes = plot_temperature_salinity_profiles(
                profiles, fid,
                ng_depth=ng_d, ng_temperature=ng_T, ng_salinity=ng_S,
                mouth_depth=mouth_d, mouth_temperature=mouth_T, mouth_salinity=mouth_S,
                sill_depth=sill_d, grounding_depth=grnd_d,
            )

            log("  Subsetting BedMachine …")
            bed_subset = subset_bedmachine(
                bed_cache,
                (ng_row["longitude"], ng_row["latitude"]),
                (mouth_df.iloc[0]["longitude"], mouth_df.iloc[0]["latitude"]),
            )

            log("  Plotting TS diagram …")
            plot_ts_diagram(
                axes[1, 0],
                ng_d, ng_T, ng_S,
                mouth_d, mouth_T, mouth_S,
                mid_d, mid_T, mid_S,
                fjord_row=ng_row,
            )

            if mid:
                log("  Plotting bathymetry panel …")
                plot_bathymetry_panel(
                    fig, axes[1, 1],
                    bed_subset,
                    mouth_df.iloc[0],
                    ng_row,
                    mid["row"],
                    shapefile_defs=shapefile_defs,
                )

            out_path = output_dir / f"{fid}_withInset_10mBins_v2.png"
            plt.savefig(out_path, dpi=150, bbox_inches="tight")  # 150 dpi saves faster
            log(f"  Saved → {out_path}")
            plt.close(fig)

        except Exception as exc:
            log(f"  ERROR ({fid}): {exc}")
            traceback.print_exc()


def main():
    for csv_file in (qim_csv, klu_csv):
        process_csv(csv_file, nc_dir, output_dir, bed_cache, SHAPEFILE_DEFS)
    log("\nAll done.")


if __name__ == "__main__":
    main()

[LOG] Loading C:\Users\efc4\OneDrive - University of St Andrews\Desktop\PhD\First_Year\Projects\Fjord_Shelf_Observations\Working_data\OMG_data\greenland\qim_fjord_data.csv …
[LOG] Filtering to 2016 profiles …
[LOG]   3 rows after 2016 filter.
[LOG] Matching NG ↔ Mouth pairs …
[LOG]   1 matched, 0 failed.
[LOG] 
=== Fjord 1/1  (id=284(2016)) ===
[LOG]   Loading and binning NetCDF files …
[LOG]   Plotting profiles …
[LOG]   Subsetting BedMachine …
[LOG]   Plotting TS diagram …


C:\Users\efc4\AppData\Local\Temp\ipykernel_18688\4011892128.py:461: MatplotlibDeprecationWarning: The collections attribute was deprecated in Matplotlib 3.8 and will be removed two minor releases later.
  contour_path     = dsow.collections[0].get_paths()[0].vertices


[LOG]   Plotting bathymetry panel …
[LOG]   Saved → C:\Users\efc4\OneDrive - University of St Andrews\Desktop\PhD\First_Year\Projects\Fjord_Shelf_Observations\Results\caseStudies_qimKlu\284(2016)_withInset_10mBins_v2.png
[LOG] Loading C:\Users\efc4\OneDrive - University of St Andrews\Desktop\PhD\First_Year\Projects\Fjord_Shelf_Observations\Working_data\OMG_data\greenland\klu_fjord_data.csv …
[LOG] Filtering to 2016 profiles …
[LOG]   3 rows after 2016 filter.
[LOG] Matching NG ↔ Mouth pairs …
[LOG]   1 matched, 0 failed.
[LOG] 
=== Fjord 1/1  (id=253(2016)) ===
[LOG]   Loading and binning NetCDF files …
[LOG]   Plotting profiles …
[LOG]   Subsetting BedMachine …
[LOG]   Plotting TS diagram …
[LOG]   Plotting bathymetry panel …


C:\Users\efc4\AppData\Local\Temp\ipykernel_18688\4011892128.py:461: MatplotlibDeprecationWarning: The collections attribute was deprecated in Matplotlib 3.8 and will be removed two minor releases later.
  contour_path     = dsow.collections[0].get_paths()[0].vertices


[LOG]   Saved → C:\Users\efc4\OneDrive - University of St Andrews\Desktop\PhD\First_Year\Projects\Fjord_Shelf_Observations\Results\caseStudies_qimKlu\253(2016)_withInset_10mBins_v2.png
[LOG] 
All done.


In [8]:
from config_loader import load_paths
import pandas as pd

paths = load_paths()
csv_dir = paths["csv_dir"]

for name, csv_file in [("qim", csv_dir / "qim_fjord_data.csv"),
                        ("klu", csv_dir / "klu_fjord_data.csv")]:
    df = pd.read_csv(csv_file)
    print(f"\n=== {name} columns ===")
    print(list(df.columns))
    print(df.head(6).to_string())


=== qim columns ===
['filename', 'latitude', 'longitude', 'instrument', 'year', 'type', 'fjord_id', 'GSW_filename', 'grounding_line_depth_m', 'sill_depth_m']
                               filename   latitude  longitude                                                  instrument  year          type   fjord_id                              GSW_filename  grounding_line_depth_m  sill_depth_m
0  OMG_Ocean_AXCTD_L2_20160915161939.nc  70.616020 -51.961151  Airborne eXpendable Conductivity Temperature Depth (AXCTD)  2016   fjord_mouth  284(2016)  OMG_Ocean_AXCTD_L2_20160915161939_gsw.nc                     NaN           NaN
1  OMG_Ocean_AXCTD_L2_20160915163322.nc  70.451576 -51.329411  Airborne eXpendable Conductivity Temperature Depth (AXCTD)  2016     mid_fjord  284(2016)  OMG_Ocean_AXCTD_L2_20160915163322_gsw.nc                     NaN           NaN
2  OMG_Ocean_AXCTD_L2_20160915164446.nc  70.366150 -50.795441  Airborne eXpendable Conductivity Temperature Depth (AXCTD)  2016  near_glacier 